In [1]:
import numpy as np
from matplotlib import pyplot as plt

In [2]:
from scipy.integrate import solve_ivp

In [72]:
M = 1
a = 0

In [73]:
def unpack(y):
    r1 = np.array([y[0],y[1],y[2]])
    v1 = np.array([y[3],y[4],y[5]])
    return r1,v1

In [74]:
r?

Object `r` not found.


In [75]:
def K(r,a=a):
    x,y,z = r[0],r[1],r[2]
    return (x**2+y**2+z**2-a**2)/2
    
def W(r,a=a):
    x,y,z = r[0],r[1],r[2]
    return np.sqrt(K(r)**2+a**2*z**2)

def rkw(r, a=a):
    return np.sqrt(K(r)+W(r))

In [101]:
def dx_K(r,a=a):
    x,y,z = r[0],r[1],r[2]
    return x
def dy_K(r,a=a):
    x,y,z = r[0],r[1],r[2]
    return y
def dz_K(r,a=a):
    x,y,z = r[0],r[1],r[2]
    return z

def di_K(r,a=a):
    return np.array([dx_K(r), dy_K(r), dz_K(r)])
    


def dx_W(r,a=a):
    x,y,z = r[0],r[1],r[2]
    return K(r)*x/W(r)

def dy_W(r,a=a):
    x,y,z = r[0],r[1],r[2]
    return K(r)*y/W(r)

def dz_W(r,a=a):
    x,y,z = r[0],r[1],r[2]
    return z*(K(r)+a**2)/W(r)

def di_W(r,a=a):
    return np.array([dx_W(r), dy_W(r), dz_W(r)])

In [108]:
def rho2(r,a=a):
    x,y,z = r[0],r[1],r[2]
    return rkw(r)**2+a**2

def dx_r(r,a=a):
    x,y,z = r[0],r[1],r[2]
    return x*rkw(r)/(2*W(r))

def dy_r(r,a=a):
    x,y,z = r[0],r[1],r[2]
    return y*rkw(r)/(2*W(r))

def dz_r(r,a=a):
    x,y,z = r[0],r[1],r[2]
    return z*rho2(r)/(2*rkw(r)*W(r))

def di_r(r,a=a):
    return np.array([dx_r(r), dy_r(r), dz_r(r)])

In [109]:
di_r(x)

array([0.26726124, 0.53452248, 0.80178373])

In [110]:
def f(r,a=a):
    x,y,z = r[0],r[1],r[2]
    return M*rkw(r)/W(r)
    
def dx_f(r,a=a):
    x,y,z = r[0],r[1],r[2]
    return M/W(r) * (dx_r(r)-rkw(r)/W(r)*dx_W(r))

def dy_f(r,a=a):
    x,y,z = r[0],r[1],r[2]
    return M/W(r) * (dy_r(r)-rkw(r)/W(r)*dy_W(r))

def dz_f(r,a=a):
    x,y,z = r[0],r[1],r[2]
    return M/W(r) * (dz_r(r)-rkw(r)/W(r)*dz_W(r))


def di_f(r,a=a):
    return np.array([dx_f(r), dy_f(r), dz_f(r)])

In [136]:
def Lx(r, a=a):
    x,y,z = r[0],r[1],r[2]
    return (x*(a**2-rkw(r)**2) - 2*a*rkw(r)*y)/(rho2(r)**2)

def Ly(r, a=a):
    x,y,z = r[0],r[1],r[2]
    return (y*(a**2-rkw(r)**2) - 2*a*rkw(r)*x)/(rho2(r)**2)

def Lz(r, a=a):
    x,y,z = r[0],r[1],r[2]
    return -z/rkw(r)**2

def L(r,a=a):
    return np.array([Lx(r), Ly(r), Lz(r)])


def l(r,a=a):
    '''indici bassi'''
    x,y,z = r[0],r[1],r[2]
    lt = 1
    lx = (rkw(r)*x+a*y)/rho2(r)
    ly = (rkw(r)*y-a*x)/rho2(r)
    lz = z/rkw(r)
    return np.array([lt, lx, ly, lz])

def di_lx(r,a=a):
    dx_lx = rkw(r)/rho2(r) + Lx(r)*dx_r(r)
    dy_lx = a/rho2(r) + Lx(r)*dy_r(r)
    dz_lx = Lx(r)*dz_r(r)
    return np.array([dx_lx, dy_lx, dz_lx])

def di_ly(r,a=a):
    dy_ly = rkw(r)/rho2(r) + Ly(r)*dy_r(r)
    dx_ly = -a/rho2(r) + Ly(r)*dx_r(r)
    dz_ly = Ly(r)*dz_r(r)
    return np.array([dx_ly, dy_ly, dz_ly])

def di_lz(r,a=a):
    dx_lz = Lz(r)*dx_r(r)
    dy_lz = Lz(r)*dy_r(r)
    dz_lz = 1/rkw(r)+Lz(r)*dz_r(r)
    return np.array([dx_lz, dy_lz, dz_lz])

def deriv_l(r,a=a):
    di_lt = np.array([0,0,0])
    Dl =  np.array([di_lt, di_lx(r), di_ly(r), di_lz(r)])
    Dl = Dl.T
    return Dl

In [137]:
#np.einsum
deriv_l(x)

array([[ 0.        ,  0.24817115, -0.03818018, -0.05727027],
       [ 0.        , -0.03818018,  0.19090089, -0.11454053],
       [ 0.        , -0.05727027, -0.11454053,  0.09545044]])

In [145]:
def deriv_gmn344(r,a=a):
    D_g = np.zeros((3,4,4))
    for k in range(3):
        for mu in range(4):
            for nu in range(4):
                D_g[k, mu, nu] = di_f(r)[k] * l(r)[mu] * l(r)[nu] + f(r) * deriv_l(r)[k,mu] * l(r)[nu] + f(r) * l(r)[mu] * deriv_l(r)[k,nu]

    return D_g

def deriv_gmn444(r,a=a):
    D_g = np.zeros((4,4,4))
    for k in range(3):
        for mu in range(4):
            for nu in range(4):
                D_g[k+1, mu, nu] = di_f(r)[k] * l(r)[mu] * l(r)[nu] + f(r) * deriv_l(r)[k,mu] * l(r)[nu] + f(r) * l(r)[mu] * deriv_l(r)[k,nu]
                D_g[0, mu, nu] = 0

    return D_g

In [153]:
def G(r,a=a):
    Galphabeta = np.zeros((4,4))
    Gabs = np.zeros((4,4,4))
    for alpha in range(4):
        for beta in range(4):
            for sigma in range(4):
                Gabs[alpha,beta,sigma] = l(r)[sigma] * ( deriv_gmn444(r)[alpha,beta,sigma] + deriv_gmn444(r)[beta,alpha,sigma] - deriv_gmn444(r)[sigma,alpha,beta])
                
            Galphabeta[alpha, beta] = np.einsum('ijk->ij', Gabs)
    return Galphabeta

In [155]:
def G(r,a=a):
    Galphabeta = np.zeros((4,4))
    #Gabs = np.zeros((4,4,4))
    for alpha in range(4):
        for beta in range(4):
            for sigma in range(4):
                Galphabeta[alpha,beta] = Galphabeta[alpha,beta] + l(r)[sigma] * ( deriv_gmn444(r)[alpha,beta,sigma] + deriv_gmn444(r)[beta,alpha,sigma] - deriv_gmn444(r)[sigma,alpha,beta] )
                
            #Galphabeta[alpha, beta] = np.einsum('k,ijk->ij', l(r), d_g)
    return Galphabeta

In [160]:
def G(r,a=a):
    ll = l(r)
    ll[0] = -ll[0]
    dg = deriv_gmn444(r)

    G = np.einsum('i, jki -> jk',  ll, dg) + np.einsum('i, kji -> jk',  ll, dg) - np.einsum('i, ijk -> jk',  ll, dg) 
    return G
    

In [161]:
x = np.array([1,2,3])
G(x)

array([[0.14285714, 0.03818018, 0.07636035, 0.11454053],
       [0.03818018, 0.01020408, 0.02040816, 0.03061224],
       [0.07636035, 0.02040816, 0.04081633, 0.06122449],
       [0.11454053, 0.03061224, 0.06122449, 0.09183673]])

In [163]:
def eta_mn(r,a=a):
    eta = np.zeros((4,4))
    eta[0,0] = -1
    for i in range(1,4):
        eta[i,i] = 1
    return eta

def B_abs(r,a=a):
    B = np.zeros((4,4,4))
    for alpha in range(4):
        for beta in range(4):
            for sigma in range(4):
                B[alpha,beta,sigma] = deriv_gmn444(r)[alpha,beta,sigma] + deriv_gmn444(r)[beta,alpha,sigma] - deriv_gmn444(r)[sigma,alpha,beta]
    return B

def Gamma_bar_mab(r,a=a):
    gamma = np.zeros((4,4,4))
    for mu in range(4):
        for alpha in range(4):
            for beta in range(4):
                for sigma in range(4):
                    gamma[mu,alpha,beta] += 0.5 * eta_mn(r)[mu,sigma] * B_abs(r)[alpha,beta,sigma]
    return gamma      

def Gamma_bar_mab(r,a=a):
    n = eta_mn(r) 
    B_2 =  0.5 * B_abs(r)
    gamma = np.einsum('ms, abs -> mab', n, B_2)
    return gamma   

In [165]:
Gamma_bar_mab(x)

array([[[ 0.        ,  0.01909009,  0.03818018,  0.05727027],
        [ 0.01909009, -0.12244898,  0.04081633,  0.06122449],
        [ 0.03818018,  0.04081633, -0.06122449,  0.12244898],
        [ 0.05727027,  0.06122449,  0.12244898,  0.04081633]],

       [[ 0.01909009,  0.        ,  0.        ,  0.        ],
        [ 0.        ,  0.03408944, -0.00818147, -0.0122722 ],
        [ 0.        , -0.00818147,  0.02181724, -0.0245444 ],
        [ 0.        , -0.0122722 , -0.0245444 ,  0.00136358]],

       [[ 0.03818018,  0.        ,  0.        ,  0.        ],
        [ 0.        ,  0.06817889, -0.01636293, -0.0245444 ],
        [ 0.        , -0.01636293,  0.04363449, -0.0490888 ],
        [ 0.        , -0.0245444 , -0.0490888 ,  0.00272716]],

       [[ 0.05727027,  0.        ,  0.        ,  0.        ],
        [ 0.        ,  0.10226833, -0.0245444 , -0.0368166 ],
        [ 0.        , -0.0245444 ,  0.06545173, -0.0736332 ],
        [ 0.        , -0.0368166 , -0.0736332 ,  0.00409073]]])

In [188]:
def G_scalar(r,v,a=a):
    Gs = np.einsum('ab,a,b -> ', G(r), v, v)
    return Gs

def dv_dt(r,v,a=a):
    v4 = np.insert(v,0,0)
    gmu_vv = np.einsum('mab, a, b -> m', Gamma_bar_mab(r), v4, v4)
    gi_vv = gmu_vv[1:]
    gmu_vvv = np.einsum('mab, a, b, n -> mn', Gamma_bar_mab(r), v4, v4, v4)
    gt_vvvi = gmu_vvv[0,1:]
    Fi = -gi_vv + gt_vvvi
    
    dvdt = Fi + f(r)*0.5 * G_scalar(r,v4) * (l(r)[1:] + v4[1:])
    return dvdt

In [189]:
v = np.array([1,2,3])
dv_dt(x,v)


array([2.94464087, 5.88928174, 8.83392261])

In [190]:
h = 0.01
tf = 25000

t_span =  [0,tf]

y0 = [50, 0, 0, 0, 0.14142136, 0]

t_ev = np.linspace(0,10000,int(1e4))

sol = solve_ivp(dv_dt, t_span, y0, t_eval=t_ev, rtol=1e-10, atol=1e-10)

TypeError: 'float' object is not subscriptable